# ScaleNorm

源码导航：[core/norm/scale_norm.py](../../../core/norm/scale_norm.py) 中的 `ScaleNorm`。

Nguyen & Chiang (2019) 在 *Transformers without Tears* 中提出 **ScaleNorm**，用 ℓ₂ 范数替代 LayerNorm 的标准差，并将所有维度的缩放压缩为**单个可学习标量** `g`。该设计在参数量与计算量上均显著低于 LayerNorm，且特别适合低资源机器翻译与隐私训练（DP-SGD）场景。

### 1. 理论推导

设 $x \in \mathbb{R}^d$ 为某一 token 的隐状态向量，其 ℓ₂ 范数为：

$$\|x\|_2 = \sqrt{\sum_{i=1}^{d} x_i^2}$$

ScaleNorm 的输出为：

$$\text{ScaleNorm}(x; g) = g \cdot \frac{x}{\|x\|_2 + \epsilon}$$

其中 $g \in \mathbb{R}$ 是唯一的可学习参数，$\epsilon$ 为防止除零的数值稳定项。与 LayerNorm 不同，ScaleNorm **不减均值、不除标准差**，而是将整个向量投影到 $(d-1)$ 维超球面上，再用标量 $g$ 控制球面半径。

**与 LayerNorm / RMSNorm 的对比：**

| 属性 | LayerNorm | RMSNorm | ScaleNorm |
|---|---|---|---|
| 中心化（减均值） | ✓ | ✗ | ✗ |
| 缩放统计量 | 标准差 $\sigma$ | RMS | ℓ₂ 范数 $\|x\|_2$ |
| 可学习参数数（dim=$d$）| $2d$（含 bias）| $d$ | **$1$** |
| 计算量（归一化部分）| $O(3d)$ | $O(2d)$ | **$O(d)$** |

当与输出层的 FixNorm 结合时，ScaleNorm 等价于 **Cosine Normalization**：内积被转化为余弦相似度，有助于防止罕见词的 logits 过度尖锐。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.scale_norm import ScaleNorm

### 2. 形状与 ℓ₂ 范数数值检查

In [2]:
torch.manual_seed(0)
x = torch.randn(2, 4, 16) * 5.0  # 放大输入以验证归一化效果
norm = ScaleNorm(16, eps=1e-6)
y = norm(x)

print("x.shape =", tuple(x.shape))
print("y.shape =", tuple(y.shape))
# g 初始化为 1.0：归一化后每 token 的 ℓ₂ 范数应接近 1
print("ℓ₂ before:", x.norm(dim=-1).mean().item())
print("ℓ₂ after :", y.norm(dim=-1).mean().item())
assert x.shape == y.shape, "ScaleNorm 必须保持输入输出维度一致！"

x.shape = (2, 4, 16)
y.shape = (2, 4, 16)
ℓ₂ before: 20.68267059326172
ℓ₂ after : 1.0


### 3. 标量参数 $g$ 的缩放效应

In [3]:
norm_g3 = ScaleNorm(16, eps=1e-6)
with torch.no_grad():
    norm_g3.g.fill_(3.0)

y_g3 = norm_g3(x)
print("g = 3.0 时 ℓ₂ after:", y_g3.norm(dim=-1).mean().item())
assert torch.allclose(y_g3.norm(dim=-1), torch.full((2, 4), 3.0), atol=1e-3)

g = 3.0 时 ℓ₂ after: 3.0


### 4. 与 LayerNorm / RMSNorm 的参数量对比

In [4]:
import torch.nn as nn

from core.norm.rmsnorm import RMSNorm

dim = 1536
scale = ScaleNorm(dim)
rms = RMSNorm(dim)
ln  = nn.LayerNorm(dim, elementwise_affine=True)

scale_params = sum(p.numel() for p in scale.parameters())
rms_params   = sum(p.numel() for p in rms.parameters())
ln_params    = sum(p.numel() for p in ln.parameters())

print(f"ScaleNorm params: {scale_params:,}  (仅单标量 g)")
print(f"RMSNorm   params: {rms_params:,}  (逐维 scale w)")
print(f"LayerNorm params: {ln_params:,}  (scale w + bias b)")
print(f"ScaleNorm 相比 LayerNorm 节省: {(1 - scale_params / ln_params) * 100:.2f}%")

ScaleNorm params: 1  (仅单标量 g)
RMSNorm   params: 1,536  (逐维 scale w)
LayerNorm params: 3,072  (scale w + bias b)
ScaleNorm 相比 LayerNorm 节省: 99.97%


### 5. 源码精讲

以下为 `core/norm/scale_norm.py` 中 `ScaleNorm` 的完整实现：

```python
class ScaleNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-6, g: float | None = None) -> None:
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        init_g = g if g is not None else 1.0
        # 唯一可学习参数：标量 g，对应公式中的球面半径
        self.g = nn.Parameter(torch.tensor(init_g, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # fp32 计算以提高数值稳定性
        orig_dtype = x.dtype
        x_fp32 = x.float()
        # 在最后一维计算 ℓ₂ 范数，clamp_min 防止除零
        norm = x_fp32.norm(dim=-1, keepdim=True).clamp_min(self.eps)
        # 投影到单位球面后再用 g 缩放
        out = x_fp32 / norm * self.g.to(x_fp32.dtype)
        return out.to(orig_dtype)
```

关键设计点：
- `norm(dim=-1, keepdim=True)` 产生形状 `(B, T, 1)`，可与 `x_fp32` `(B, T, d)` 广播相除。
- `g` 为标量参数，全局控制所有特征维度的输出尺度，不引入逐维偏置。
- `clamp_min(self.eps)` 等价于公式中的 $+ \epsilon$，确保全零输入时不会除零崩溃。

---

## 延伸阅读与参考资料

### 核心论文
- **Transformers without Tears: Improving the Normalization of Self-Attention**: Nguyen and Chiang, 2019. [arXiv:1910.05895](https://arxiv.org/abs/1910.05895)

### 相关实现
- **GPT-Neo ScaleNorm**: [EleutherAI/gpt-neo](https://github.com/EleutherAI/gpt-neo)
- **FLASH Transformer**: 默认使用 ScaleNorm 以加速训练